# dev: save / load / export_h5ad

Checks for `pdata.save()`, `pAnnData.load()` and `pdata.export_h5ad()` on a PD import and a DIA-NN import.

For each import: save the fresh object → load → compare, then run a chain of analysis steps → save → load → compare,
then keep working on the **loaded** object to make sure it still behaves like the original.

`same(a, b)` uses the strict comparator from the test suite (`tests/conftest.py::assert_pdata_equal`): every AnnData slot,
`.rs`, `.summary`, `.stats` (incl. `df.attrs`), `.history`, `.metadata`, with types. It raises on the first mismatch.

Output files go to `dev/saveload_out/` (delete when done).

In [ ]:
from scpviz import pAnnData as pAnnData
from scpviz import plotting as scplt
from scpviz import utils as scutils
import scanpy as sc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(context='paper', style='ticks')

import os, sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'tests'))
from conftest import assert_pdata_equal

out = Path('saveload_out')
out.mkdir(exist_ok=True)
for f in out.iterdir():   # start clean so the overwrite checks behave the same on rerun
    f.unlink()

def same(a, b, history_tail=1):
    # history_tail=1: the loaded object has one extra 'Loaded from ...' history line
    assert_pdata_equal(a, b, history_tail=history_tail)
    print('✅ identical (apart from the last', history_tail, 'history line)')

def size(path):
    print(f'{path}: {os.path.getsize(path) / 1e6:.1f} MB')

# PD import (`pd_prot.txt` + `pd_pep.txt`)

In [ ]:
obs_columns = ['sample', 'cellline', 'treatment']
pdata_pd = pAnnData.import_proteomeDiscoverer(prot_file='pd_prot.txt', pep_file='pd_pep.txt', obs_columns=obs_columns)

## 1. Fresh import → save → load

In [ ]:
path = pdata_pd.save(out / 'pd_fresh')   # no extension → '.pdata' is appended
size(path)

In [ ]:
pd_loaded = pAnnData.load(out / 'pd_fresh')   # extension optional here too
pd_loaded

In [ ]:
same(pdata_pd, pd_loaded)
pd_loaded.metadata

## 2. Path rules: refuse to overwrite, `overwrite=True`

In [ ]:
try:
    pdata_pd.save(out / 'pd_fresh')
except FileExistsError as e:
    print('raised as expected:', e)

pdata_pd.save(out / 'pd_fresh', overwrite=True)

## 3. Analysis chain on PD

In [ ]:
pdata_pd = pdata_pd.filter_prot(condition='unique_peptides > 1')  # return_copy=False crashes with peptides (todo bug-10)
# annotate after filtering: found metrics aren't subset by later filters (todo bug-11)
pdata_pd.annotate_found(classes='cellline', on='protein')
pdata_pd.annotate_found(classes='cellline', on='peptide')
pdata_pd.normalize(method='median')
pdata_pd.impute(classes=['cellline', 'treatment'], method='mean')
pdata_pd.pca(on='protein')
pdata_pd.neighbor(on='protein')
pdata_pd.umap(on='protein')
pdata_pd.leiden(on='protein', resolution=0.5)

In [ ]:
values = [{'cellline': 'AS', 'treatment': 'kd'}, {'cellline': 'AS', 'treatment': 'sc'}]
de_mean = pdata_pd.de(values)
de_peppair = pdata_pd.de(values, fold_change_mode='pep_pairwise_median')
pdata_pd.de([{'cellline': 'BE'}, {'cellline': 'AS'}])
[k for k in pdata_pd.stats if ' vs ' in k]

In [ ]:
pdata_pd.pairwise_correlation(classes='cellline')
pdata_pd.cv(classes='cellline')
pdata_pd.rank(classes='cellline')
g = scplt.plot_clustered_heatmap(pdata_pd, classes=['cellline', 'treatment'],
                                 proteins=pdata_pd.prot.var['Genes'].dropna().astype(str).unique()[:30].tolist())
plt.close('all')

In [ ]:
# needs internet (STRING)
genelist = ['P55072', 'NPLOC4', 'UFD1', 'STX5A', 'NSFL1C', 'UBXN2A', 'UBXN4', 'UBE4B', 'YOD1', 'WASHC5', 'PLAA', 'UBXN10']
pdata_pd.enrichment_functional(genes=genelist, from_de=False)
pdata_pd.enrichment_ppi(genes=genelist)
pdata_pd.list_enrichments()

In [ ]:
pdata_pd.update_identifier_maps({pdata_pd.prot.var_names[0]: 'NEWGENE'}, on='protein', direction='reverse', overwrite=True)
pdata_pd.print_history()

## 4. Save → load after the chain

In [ ]:
path = pdata_pd.save(out / 'pd_chain')
size(path)
pd_loaded = pAnnData.load(path)
same(pdata_pd, pd_loaded)

In [ ]:
pd_loaded.print_history()

In [ ]:
pd_loaded.list_enrichments()
pd_loaded.stats['functional']['UserSearch1']['result'].head()

## 5. Keep working on the loaded object

In [ ]:
key = "[{'cellline': 'BE'}] vs [{'cellline': 'AS'}]"
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
scplt.plot_volcano(axes[0], pdata=pdata_pd, stats_key=key); axes[0].set_title('original')
scplt.plot_volcano(axes[1], pdata=pd_loaded, stats_key=key); axes[1].set_title('loaded')
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
scplt.plot_umap(axes[0], pdata_pd, color='leiden'); axes[0].set_title('original')
scplt.plot_umap(axes[1], pd_loaded, color='leiden'); axes[1].set_title('loaded')
plt.tight_layout()

In [ ]:
# fresh DE on the loaded object matches the DE stored before saving
fresh = pd_loaded.de([{'cellline': 'BE'}, {'cellline': 'AS'}])
numeric = ['log2fc', 'p_value', 'test_statistic']
pd.testing.assert_frame_equal(fresh[numeric], pdata_pd.stats[key][numeric])
print('✅ fresh de() matches')

In [ ]:
# filter using the saved 'found in' metrics; peptide-level DE needs the RS + metadata['source']
filtered = pd_loaded.filter_prot_found(group=['AS', 'BE'], min_count=3)
pd_loaded.de(values, fold_change_mode='pep_pairwise_median').head()

In [ ]:
# loaded object → copy → save → load still round-trips
pd_loaded2 = pAnnData.load(filtered.save(out / 'pd_filtered', overwrite=True))
same(filtered, pd_loaded2)

## 6. `export_h5ad` (for scanpy users; drops the other modality, rs, summary, history)

In [ ]:
h5 = pdata_pd.export_h5ad(out / 'pd_prot', on='protein', overwrite=True)
h5_pep = pdata_pd.export_h5ad(out / 'pd_pep', on='peptide', overwrite=True)

In [ ]:
adata = sc.read_h5ad(h5)
print(adata)
print(adata.uns['scpviz'].keys())
print(adata.uns['scpviz']['metadata'])

# DIA-NN import (`tests/test_diann.parquet`)

In [ ]:
obs_columns = ['name', 'date', 'MS', 'acquisition', 'FAIMS', 'column', 'gradient', 'amt', 'region', 'replicate']
pdata_diann = pAnnData.import_diann(str(Path('..') / 'tests' / 'test_diann.parquet'), obs_columns=obs_columns)
pdata_diann.prot.obs[['gradient', 'amt', 'replicate']].value_counts()

## 1. Fresh import → save → load

In [ ]:
path = pdata_diann.save(out / 'diann_fresh', overwrite=True)
size(path)
diann_loaded = pAnnData.load(path)
same(pdata_diann, diann_loaded)
diann_loaded

## 2. Analysis chain on DIA-NN

In [ ]:
pdata_diann.annotate_significant(classes=['amt'], fdr_threshold=0.05, on='protein')
pdata_diann.annotate_found(classes='gradient', on='protein')
pdata_diann.normalize(method='median')
pdata_diann.impute(method='mean')
pdata_diann.pca(on='protein')
pdata_diann.de([{'gradient': '24min'}, {'gradient': '60min'}])
pdata_diann.pairwise_correlation(classes='gradient')

In [ ]:
# mixed model (stores per-contrast frames with df.attrs + a collection)
mixed = pdata_diann.mixed_de(group_col='gradient', contrast=('24min', '60min'), donor_col='replicate', store=True,
                             observation_level='pseudobulk', min_detected_fraction=0.0, min_cells_detected=1)
[k for k in pdata_diann.stats]

## 3. Save → load after the chain

In [ ]:
path = pdata_diann.save(out / 'diann_chain', overwrite=True)
size(path)
diann_loaded = pAnnData.load(path)
same(pdata_diann, diann_loaded)
diann_loaded.print_history()

In [ ]:
# df.attrs from mixed_de survive
k = [k for k in pdata_diann.stats if 'mixed' in str(k).lower() or isinstance(pdata_diann.stats[k], pd.DataFrame) and pdata_diann.stats[k].attrs]
for key_ in k:
    v = diann_loaded.stats[key_]
    if isinstance(v, pd.DataFrame):
        print(key_, '→ attrs keys:', list(v.attrs))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
scplt.plot_pca(axes[0], pdata_diann, color='gradient'); axes[0].set_title('original')
scplt.plot_pca(axes[1], diann_loaded, color='gradient'); axes[1].set_title('loaded')
plt.tight_layout()

In [ ]:
dkey = [k for k in pdata_diann.stats if ' vs ' in k][0]
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
scplt.plot_volcano(axes[0], pdata=pdata_diann, stats_key=dkey); axes[0].set_title('original')
scplt.plot_volcano(axes[1], pdata=diann_loaded, stats_key=dkey); axes[1].set_title('loaded')
plt.tight_layout()

## 4. `export_h5ad`

In [ ]:
h5 = pdata_diann.export_h5ad(out / 'diann_prot', on='protein', overwrite=True)
adata = sc.read_h5ad(h5)
print(adata)
print(adata.uns['scpviz'].keys())

# Hand-built object + `load(source=...)`

A pAnnData built by hand has no import source, so peptide-level functions (`pep_pairwise_median`) can't map peptides.
`load` warns about it; `source=` fills it in, or set `pdata.metadata['source']` yourself.

In [ ]:
hand = pAnnData(prot=pdata_pd.prot.copy(), pep=pdata_pd.pep.copy(), rs=pdata_pd.rs.copy())
hand.update_summary(verbose=False)   # constructor leaves .summary empty (todo bug-12)
hand.metadata

In [ ]:
hand.save(out / 'hand', overwrite=True)
hand_loaded = pAnnData.load(out / 'hand')           # warns: no source recorded

In [ ]:
hand_loaded = pAnnData.load(out / 'hand', source='proteomeDiscoverer')
print(hand_loaded.metadata)
hand_loaded.de(values, fold_change_mode='pep_pairwise_median').head()

# Cleanup

In [ ]:
# shutil.rmtree(out)